# 01 · Data Build
**Chicago Energy Burden project, Stage 1**

This notebook downloads the raw data, computes household energy burden for every census tract in Chicago, joins neighborhood characteristics from the American Community Survey, and checks the results against published benchmarks.

**To run it:** choose **Runtime > Run all** from the menu. It takes a few minutes, mostly for downloads.

## Setup
In Google Colab this cell copies the project code from GitHub and installs the few libraries Colab doesn't already have. On your own computer it just finds the project folder.

In [ ]:
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("Chicago-Energy-Burden").exists():
        !git clone -q https://github.com/dayyan24-netizen/Chicago-Energy-Burden.git
    os.chdir("Chicago-Energy-Burden")
    !pip install -q -r requirements.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, "src")
from ceb import config, download, build
print("Project folder:", Path.cwd())

## Step 1: Download the raw data
Three sources, each saved under `data/raw/`. Files that are already there are skipped.

| Source | Why we need it |
|---|---|
| DOE LEAD Tool (Illinois, 2022) | Household income and energy costs by tract |
| Census TIGER/Line boundaries | Tract shapes and the Chicago city limit |
| American Community Survey 2018-2022 | Neighborhood characteristics with margins of error |

In [ ]:
lead_zip = download.download_lead()
tract_zip, place_zip = download.download_tiger()
acs_cov, acs_cost = download.download_acs()

## Step 2: Build the tract dataset
This step keeps only tracts inside Chicago, adds up LEAD's household groups into one burden figure per tract, builds the ACS percentages, flags unreliable tracts, and saves spatial weights for later models.

In [ ]:
df = build.run(tract_zip, place_zip, lead_zip, acs_cov, acs_cost)
print(f"{len(df)} tracts, {df['analysis_sample'].sum()} in the analysis sample")

## Step 3: Check the results
**Read this before moving on.** The key line is the citywide burden. It should land near ACEEE's published 2-3%. If the report shows a warning, the LEAD cost columns may be in different units than expected, and we fix that before any modeling.

In [ ]:
from IPython.display import Markdown, Image, display
report = (config.REPORTS / "validation_report.md").read_text()
display(Markdown(report.split("## Map")[0]))
display(Image(str(config.REPORTS / "figures" / "burden_map.png"), width=500))

## Step 4: Explore the data
Summary statistics for the main variables:

In [ ]:
cols = ["burden", "share_high", "share_severe", "median_hh_income", "pct_black",
        "pct_hispanic", "pct_pre1950", "pct_renter", "pct_multifamily"]
df.loc[df["analysis_sample"], cols].describe().round(2).T

Interactive map: hover over any tract to see its numbers. Zoom and pan like Google Maps.

In [ ]:
df[df["analysis_sample"]].explore(
    column="burden", scheme="quantiles", k=5, cmap="YlOrRd",
    tooltip=["GEOID", "burden", "median_hh_income", "pct_pre1950", "pct_renter"],
    legend_kwds={"caption": "Energy burden (% of income)"},
    tiles="CartoDB positron")

## What's next
Stage 2 tests whether burden is spatially clustered (global Moran's I) and finds the specific hotspots (LISA cluster maps). The dataset and spatial weights saved here are its inputs.